# 🔬 Explainable AI (XAI) Lab: Diabetes Prediction with SHAP & LIME
**Modular Laboratory Workflow**  
This notebook demonstrates an end-to-end, leak-free explainable machine learning pipeline utilizing the modular `src` architecture.

### Workflow Stages:
1. **Data Ingestion & Stratified Splitting** (`src.data`)
2. **Leak-Free Pipeline Training & Evaluation** (`src.model`)
3. **Global Model Interpretability with SHAP** (`src.explainers.shap_engine`)
4. **Local & Borderline Patient Diagnosis with Unified PatientExplainer** (`src.explainers`)

## Step 1 — Ingestion & Leakage-Free Preprocessing

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
root_path = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import pandas as pd
from src.data import load_raw_data, clean_missing_values, get_train_test_split, FEATURE_NAMES

# 1. Ingest raw data with local fallback
raw_df = load_raw_data()
cleaned_df = clean_missing_values(raw_df)

print(f"Dataset shape        : {cleaned_df.shape}")
print(f"Diabetic prevalence  : {cleaned_df['Outcome'].mean():.2%}")
print(f"Features             : {', '.join(FEATURE_NAMES)}")

# 2. Stratified train/test split (Zero-leakage: imputation deferred to pipeline)
X_train, X_test, y_train, y_test = get_train_test_split(cleaned_df, test_size=0.2, random_state=42)
print(f"Training instances   : {len(X_train)} rows")
print(f"Testing instances    : {len(X_test)} rows")
cleaned_df.head()

## Step 2 — Model Pipeline Training & Evaluation
A `scikit-learn` `Pipeline` combines `SimpleImputer(strategy='median')` with `RandomForestClassifier`. Medians are fitted exclusively on `X_train`.

In [ ]:
from src.model import build_pipeline, train_pipeline, evaluate_pipeline

# Build leak-free pipeline
pipeline = build_pipeline(n_estimators=100, max_depth=None, random_state=42)

# Train strictly on training split
pipeline = train_pipeline(pipeline, X_train, y_train)

# Evaluate on held-out test split
metrics = evaluate_pipeline(pipeline, X_test, y_test)
print(f"✅ Accuracy : {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.1f}%)")
print(f"✅ ROC-AUC  : {metrics['roc_auc']:.4f}")
print("\nDetailed Classification Report:")
print(metrics['classification_report_text'])

## Step 3 — Global Explainability (SHAP)
Global explainability uncovers which clinical indicators dominate model decision-making across the entire cohort.

In [ ]:
import matplotlib.pyplot as plt
from src.explainers.shap_engine import TreeSHAPEngine

# Initialize SHAP engine from fitted pipeline
shap_engine = TreeSHAPEngine(pipeline, feature_names=FEATURE_NAMES)
shap_values = shap_engine.compute_shap_values(X_test)

# 1. Global Summary (Beeswarm) Plot
shap_engine.plot_summary(shap_values, X_test, plot_type='dot', show=True)

# 2. Mean Absolute Importance Ranking
fig, importance_df = shap_engine.plot_feature_importance_bar(shap_values, show=True)
print("Top 3 driving features:")
print(importance_df.tail(3)[['Feature', 'Mean |SHAP|']].to_string(index=False))

# 3. Dependence Plot for Glucose (Interactions)
shap_engine.plot_dependence('Glucose', shap_values, X_test, show=True)

## Step 4 — Local & Borderline Patient Diagnosis (Unified PatientExplainer)
We analyze high-uncertainty patients near the decision boundary (|P(y=1) - 0.5|) using both SHAP and LIME.

In [ ]:
from src.explainers.visualization import find_most_uncertain_patient
from src.explainers.patient_explainer import PatientExplainer

# Extract imputed training distribution for LIME background statistics
imputer = pipeline.named_steps['imputer']
X_train_imputed = imputer.transform(X_train)

# Instantiate Unified PatientExplainer
patient_explainer = PatientExplainer(
    pipeline_or_model=pipeline,
    training_data=X_train_imputed,
    feature_names=FEATURE_NAMES,
    random_state=42
)

# Detect the most uncertain patient in the test set
uncertain_info = find_most_uncertain_patient(pipeline, X_test, y_test=y_test)
hardest_idx = uncertain_info['index']
hardest_features = uncertain_info['features']

print(f"Borderline Patient Index : #{hardest_idx}")
print(f"Predicted Probability    : {uncertain_info['prob_diabetes']:.4f}")
print(f"Ground Truth Label       : {'Diabetic' if uncertain_info['true_label'] == 1 else 'Non-Diabetic'}")

# Generate standardized multi-method diagnostic report
report = patient_explainer.explain_patient(hardest_features)
print(f"Predicted Class          : {report['predicted_class']}")
print(f"Top SHAP Driver          : {report['top_shap_feature']}")
print(f"Top LIME Condition       : {report['top_lime_condition']}")
print(f"Method Concordance       : {'✅ In Agreement' if report['top_feature_agreement'] else '⚠️ Disagreement'}")

# Side-by-side comparative visualization
fig, _ = patient_explainer.plot_comparison(hardest_idx, hardest_features, show=True)